# 3. Pattern memory, as hysteresis

Ramp a channel blocker up until a depolarized cell switches, then ramp it back down.
The cell does not return. That is a bioelectric pattern memory, measured.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tlcdv/bioelectrics/blob/main/notebooks/03-pattern-memory.ipynb)

In [ ]:
# On Colab this installs the library. Locally it does nothing if it is already there.
try:
    import bioelectrics  # noqa: F401
except ImportError:
    %pip install -q git+https://github.com/tlcdv/bioelectrics.git

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from bioelectrics.constants import DTYPE
from bioelectrics.ionic import IonicTissue, ChannelExpression, K_IR_CAM, CAT_DEP_CAM, Modifiers
from bioelectrics.attractors import dose_response

cell = IonicTissue(edges=torch.zeros(2, 0, dtype=torch.long),
                   channels=[ChannelExpression(K_IR_CAM, 1e-16),
                             ChannelExpression(CAT_DEP_CAM, 1e-16)])

def at(b):
    return Modifiers(channel_scale={'Cat_dep(CAM)': torch.tensor(1 - b, dtype=DTYPE)})

def start(v):
    return cell.with_vmem(cell.initial_state((1,)), torch.full((1, 1), v, dtype=DTYPE))

In [ ]:
doses = np.linspace(0.0, 0.9, 19)
up, _ = dose_response(cell, start(-0.003), at, doses)           # from the depolarized state
down, _ = dose_response(cell, start(-0.078), at, doses[::-1])   # from the polarized state

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(100 * doses, 1e3 * up[:, 0], lw=2, label='ramp up from depolarized')
ax.plot(100 * doses, 1e3 * down.flip(0)[:, 0], lw=2, ls='--', label='ramp down from polarized')
ax.set_xlabel('block of the depolarizing channel (%)')
ax.set_ylabel('steady-state V_mem (mV)')
ax.legend(frameon=False)
plt.show()

Where the two branches differ, the cell has two stable states and its history decides
which one it is in. Removing the drug does not undo the switch.